# Pelajaran 16 - Mendeploy Agen Skala Besar dengan Microsoft Foundry

Dalam notebook ini Anda membangun **agen dukungan pelanggan siap produksi** untuk perusahaan fiktif **Contoso**. Berbeda dengan pelajaran sebelumnya, tujuannya di sini bukan pada loop penalaran agen — melainkan semuanya yang dibungkus *di sekitar* itu yang membuat agen aman dijalankan dalam skala besar:

1. **Pemanggilan alat** — pencarian pesanan dan pembuatan tiket.
2. **RAG** — jawaban kebijakan dari basis pengetahuan.
3. **Memori** — mengingat pelanggan antar putaran.
4. **Pengarahan model** — kirim permintaan sederhana ke model kecil, yang kompleks ke model besar.
5. **Caching respons** — melayani pertanyaan berulang tanpa panggilan model.
6. **Persetujuan manusia** — pengembalian dana di atas ambang tahan menunggu tanda tangan.
7. **Gerbang evaluasi** — set uji offline yang memblokir rilis buruk.
8. **Observabilitas** — pelacakan OpenTelemetry di sekitar setiap permintaan.

Setiap bagian bersifat mandiri dan dapat dijalankan. Bacalah setiap baris — primitif produksi sengaja dibuat kecil.


## Pengaturan

Sebelum menjalankan notebook ini, pastikan Anda memiliki:

1. **Proyek Microsoft Foundry** dengan model chat yang sudah diterapkan (misalnya `gpt-4.1-mini`).
2. **Masuk dengan Azure CLI** — jalankan `az login` di terminal Anda.
3. **Atur variabel lingkungan yang dibutuhkan:**
   - `AZURE_AI_PROJECT_ENDPOINT` — endpoint proyek Microsoft Foundry Anda.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — nama model yang Anda terapkan.

Bagian RAG menggunakan **Azure AI Search** saat `AZURE_SEARCH_SERVICE_ENDPOINT` dan `AZURE_SEARCH_API_KEY` diatur, dan akan kembali menggunakan pencarian dalam memori sehingga notebook dapat berjalan tanpa sumber daya Search.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import re
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential(),
)
print("Foundry client ready.")

## 1. Alat

Alat produksi melakukan pekerjaan nyata terhadap sistem nyata. Di sini kami mensimulasikan database pesanan dan sistem tiket dengan fungsi Python biasa. Dekorator `@tool` mengeksposnya ke agen.

Perhatikan `issue_refund` menggunakan `approval_mode="always_require"` untuk pengembalian dana di atas ambang batas — ini adalah primitif human-in-the-loop yang kami terapkan nanti.


In [ ]:
# Simulated backend systems (in production these are API calls behind scoped identities).
ORDERS = {
    "A1001": {"status": "shipped", "total": 42.00, "eta": "2 days"},
    "A1002": {"status": "processing", "total": 128.50, "eta": "5 days"},
    "A1003": {"status": "delivered", "total": 19.99, "eta": "delivered"},
}
TICKETS: list[dict] = []
REFUND_APPROVAL_THRESHOLD = 50.0


@tool(approval_mode="never_require")
def get_order_status(order_id: Annotated[str, "The customer's order ID, e.g. A1001"]) -> str:
    """Look up the status of a customer order."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return f"No order found with ID {order_id}."
    return (
        f"Order {order_id.upper()}: status={order['status']}, "
        f"total=${order['total']:.2f}, eta={order['eta']}."
    )


@tool(approval_mode="never_require")
def open_ticket(
    subject: Annotated[str, "Short subject line for the support ticket"],
    details: Annotated[str, "Full description of the customer's issue"],
) -> str:
    """Open a support ticket for issues that need human follow-up."""
    ticket_id = f"T{1000 + len(TICKETS) + 1}"
    TICKETS.append({"id": ticket_id, "subject": subject, "details": details})
    return f"Ticket {ticket_id} opened: {subject}"


def refund_needs_approval(amount: float) -> bool:
    """Refunds above the threshold require a human approver."""
    return amount > REFUND_APPROVAL_THRESHOLD


@tool(approval_mode="always_require")
def issue_refund(
    order_id: Annotated[str, "The order to refund"],
    amount: Annotated[float, "Refund amount in USD"],
) -> str:
    """Issue a refund. Execution pauses for human approval before it runs."""
    return f"Refund of ${amount:.2f} issued for order {order_id.upper()}."


print("Tools defined.")

## 2. RAG — Basis Pengetahuan Kebijakan

Pertanyaan kebijakan ("berapa lama jendela pengembalian Anda?") harus dijawab dari sumber yang berwenang, bukan dari memori model. Kami membungkus basis pengetahuan kecil sebagai alat pencarian.

Dalam produksi ini adalah **Azure AI Search**; di sini kami menyediakan pencarian kata kunci dalam memori sehingga notebook dapat dijalankan di mana saja, dan beralih secara otomatis ke Azure AI Search saat variabel lingkungan tersedia.


In [ ]:
KNOWLEDGE_BASE = {
    "returns": "Contoso accepts returns within 30 days of delivery for a full refund. Items must be unused and in original packaging.",
    "shipping": "Standard shipping takes 3-5 business days. Express shipping (1-2 days) is available at checkout for an extra fee.",
    "warranty": "All Contoso electronics carry a 12-month limited warranty covering manufacturing defects.",
    "refund_policy": "Refunds are processed to the original payment method within 5 business days of approval. Refunds over $50 require a supervisor's approval.",
}


def _in_memory_search(query: str) -> str:
    q = query.lower()
    hits = [text for key, text in KNOWLEDGE_BASE.items() if key.replace("_", " ") in q or key in q]
    if not hits:
        # crude keyword fallback so the tool still returns something useful
        hits = [text for text in KNOWLEDGE_BASE.values() if any(w in text.lower() for w in q.split())]
    return "\n".join(hits) if hits else "No matching policy found."


def _azure_search(query: str) -> str:
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient

    client = SearchClient(
        endpoint=os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"],
        index_name=os.getenv("AZURE_SEARCH_INDEX_NAME", "contoso-policies"),
        credential=AzureKeyCredential(os.environ["AZURE_SEARCH_API_KEY"]),
    )
    results = client.search(search_text=query, top=3)
    return "\n".join(r.get("content", "") for r in results) or "No matching policy found."


USE_AZURE_SEARCH = bool(os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT") and os.getenv("AZURE_SEARCH_API_KEY"))


@tool(approval_mode="never_require")
def search_policies(query: Annotated[str, "The policy question to look up"]) -> str:
    """Search Contoso support policies to answer customer questions."""
    if USE_AZURE_SEARCH:
        return _azure_search(query)
    return _in_memory_search(query)


print(f"RAG ready. Using {'Azure AI Search' if USE_AZURE_SEARCH else 'in-memory search'}.")

## 3. Memori

Seorang agen dukungan yang lupa siapa yang diajak bicara adalah agen dukungan yang buruk. Kami menyimpan profil kecil per pelanggan dan menyuntikkan ringkasan singkat ke dalam instruksi agen. Dalam produksi ini adalah layanan memori (lihat Pelajaran 13); di sini sebuah dict membuat pola ini terlihat.


In [ ]:
CUSTOMER_MEMORY: dict[str, dict] = {
    "cust-42": {"name": "Dana", "tier": "enterprise", "recent_order": "A1002"},
    "cust-99": {"name": "Sam", "tier": "standard", "recent_order": "A1003"},
}


def memory_context(customer_id: str) -> str:
    profile = CUSTOMER_MEMORY.get(customer_id)
    if not profile:
        return "This is a new customer with no history."
    return (
        f"Customer {profile['name']} ({profile['tier']} tier). "
        f"Most recent order: {profile['recent_order']}."
    )


print(memory_context("cust-42"))

## 4 & 5. Routing Model dan Caching Respons

Dua pengungkit biaya yang terhubung ke satu pengelola permintaan:

- **Routing**: pengklasifikasi heuristik yang murah memutuskan apakah permintaan membutuhkan model kecil atau besar.
- **Caching**: pertanyaan yang diulang dan dinormalisasi disajikan langsung dari cache tanpa panggilan model.

Pengklasifikasi di sini sengaja dibuat sederhana. Dalam produksi, Anda akan memvalidasinya terhadap lalu lintas dan dapat menggantinya dengan Model Router Foundry.


In [ ]:
SMALL_MODEL = os.getenv("AZURE_AI_SMALL_MODEL", model)   # e.g. gpt-4.1-mini
LARGE_MODEL = os.getenv("AZURE_AI_LARGE_MODEL", model)   # e.g. gpt-4.1

response_cache: dict[str, str] = {}
route_counters = {"small": 0, "large": 0, "cache": 0}


def normalize(query: str) -> str:
    return re.sub(r"\s+", " ", query.lower().strip())


COMPLEX_SIGNALS = ("refund", "cancel", "complaint", "escalate", "broken", "wrong", "why")


def is_simple(query: str) -> bool:
    """Route complex or high-stakes requests to the large model; everything else to the small one."""
    q = query.lower()
    if any(signal in q for signal in COMPLEX_SIGNALS):
        return False
    return len(q.split()) <= 20


def choose_model(query: str) -> str:
    return SMALL_MODEL if is_simple(query) else LARGE_MODEL


print(f"Small model: {SMALL_MODEL} | Large model: {LARGE_MODEL}")

## 6 & 8. Agen, Persetujuan Manusia, dan Observabilitas

Sekarang kita merakit agen dari alat-alat di atas dan membungkus setiap permintaan dalam sebuah span OpenTelemetry. Fungsi `handle_support_request` adalah penangkap permintaan produksi: cache → route → trace → run → cache.

Persetujuan manusia diatur oleh kerangka kerja: karena `issue_refund` adalah `approval_mode="always_require"`, proses berjalan berhenti dan menampilkan permintaan persetujuan yang harus kita selesaikan secara eksplisit.


In [ ]:
# Tracing: use the Agent Framework tracer if available, else a no-op so the notebook runs anywhere.
try:
    from agent_framework.observability import get_tracer
    tracer = get_tracer()
except Exception:  # observability extras not installed
    from contextlib import contextmanager

    class _NoopSpan:
        def set_attribute(self, *_args, **_kwargs):
            pass

    class _NoopTracer:
        @contextmanager
        def start_as_current_span(self, _name):
            yield _NoopSpan()

    tracer = _NoopTracer()


SUPPORT_INSTRUCTIONS = (
    "You are Contoso's customer support agent. Be concise, friendly, and accurate. "
    "Use search_policies for policy questions, get_order_status for orders, "
    "open_ticket when a human needs to follow up, and issue_refund for refunds. "
    "Never invent policy details."
)

support_agent = provider.as_agent(
    name="ContosoSupportAgent",
    instructions=SUPPORT_INSTRUCTIONS,
    tools=[get_order_status, open_ticket, search_policies, issue_refund],
)


async def handle_support_request(query: str, customer_id: str) -> str:
    # 1. Serve from cache when we can.
    key = normalize(query)
    if key in response_cache:
        route_counters["cache"] += 1
        return response_cache[key]

    # 2. Route by complexity to control cost.
    chosen_model = choose_model(query)
    route_counters["small" if chosen_model == SMALL_MODEL else "large"] += 1

    # 3. Add per-customer memory to the prompt.
    context = memory_context(customer_id)
    prompt = f"[Customer context: {context}]\n\n{query}"

    # 4. Run inside a trace span for observability.
    with tracer.start_as_current_span("support_request") as span:
        span.set_attribute("customer.id", customer_id)
        span.set_attribute("routed.model", chosen_model)
        response = await support_agent.run(prompt, model=chosen_model)

    text = response.text
    response_cache[key] = text
    return text


print("Support agent assembled.")

In [ ]:
# Try a few requests. The first is simple (small model), the second is a refund (large model + approval path).
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print(await handle_support_request("Where is my order A1002?", "cust-42"))
print("---")
# Repeat the first question -> served from cache.
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print("Routing counters:", route_counters)

## 7. Gerbang Evaluasi

Ini adalah gerbang rilis dari pelajaran: satu set pengujian offline menilai agen, dan penerapan hanya dilanjutkan jika tingkat kelulusan melewati ambang batas. Penilai di sini adalah pemeriksaan kesamaan kata kunci sederhana untuk menjaga notebook tetap mandiri; dalam produksi Anda akan menggunakan LLM-sebagai-hakim atau evaluator kerangka kerja (lihat Pelajaran 10).


In [ ]:
TEST_CASES = [
    {"input": "How long do I have to return an item?", "expected": ["30 days", "refund"]},
    {"input": "How fast is standard shipping?", "expected": ["3-5", "business days"]},
    {"input": "What is the status of order A1001?", "expected": ["shipped", "A1001"]},
    {"input": "Do your electronics have a warranty?", "expected": ["12-month", "warranty"]},
]


def score_response(actual: str, expected_keywords: list[str]) -> float:
    actual_l = actual.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in actual_l)
    return hits / len(expected_keywords)


async def evaluation_gate(test_cases: list[dict], threshold: float = 0.8) -> bool:
    passed = 0
    for case in test_cases:
        result = await support_agent.run(case["input"])
        s = score_response(result.text, case["expected"])
        status = "PASS" if s >= 0.5 else "FAIL"
        print(f"[{status}] {case['input']}  (score={s:.0%})")
        if s >= 0.5:
            passed += 1
    pass_rate = passed / len(test_cases)
    print(f"\nEvaluation pass rate: {pass_rate:.0%} (gate: {threshold:.0%})")
    return pass_rate >= threshold


gate_passed = await evaluation_gate(TEST_CASES, threshold=0.8)
print("\nDeploy allowed:" , gate_passed)

## Menggabungkannya: Rilis Simulasi

Sel di bawah ini menampilkan keseluruhan loop yang dijelaskan oleh pelajaran: jalankan gerbang evaluasi, dan hanya "deploy" jika lulus. Ini adalah pola yang akan Anda jalankan di CI sebelum mempromosikan versi agen ke Foundry Agent Service.


In [ ]:
async def release(test_cases: list[dict]) -> None:
    print("Running pre-deployment evaluation gate...\n")
    if await evaluation_gate(test_cases, threshold=0.8):
        print("\n✅ Gate passed — promoting agent version to the Foundry Agent Service.")
    else:
        print("\n❌ Gate failed — release blocked. Fix the agent and re-run.")


await release(TEST_CASES)

## Ringkasan

Anda telah merakit agen dukungan pelanggan siap produksi dengan setiap perhatian operasional terhubung:

- **Alat, RAG, dan memori** memberi agen kemampuan dan konteks.
- **Pengalihan model dan caching** menjaga latensi dan biaya tetap terkendali.
- **Persetujuan manusia** melindungi tindakan berisiko tinggi seperti pengembalian dana besar.
- **Gerbang evaluasi** memblokir rilis buruk sebelum dikirim.
- **Pelacakan** membuat setiap permintaan dapat diamati.

### Tantangan

Perluas agen ini untuk:

1. **Mendukung banyak model** — tambahkan tingkat "penalaran" ketiga dan alihkan eskalasi/keluhan ke tingkat tersebut.
2. **Tambahkan gerbang evaluasi** — perluas `TEST_CASES` untuk memasukkan skenario persetujuan pengembalian dana dan konfirmasi gerbang menangkap regresi.
3. **Tambahkan pengalihan sadar biaya** — lacak perkiraan biaya per permintaan (kecil vs besar vs cache) dan cetak laporan biaya setelah batch kueri campuran.

Pada pelajaran berikutnya Anda mengambil perjalanan sebaliknya dan menjalankan agen sepenuhnya di mesin Anda sendiri dengan Microsoft Foundry Local dan Qwen.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan layanan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Meskipun kami berupaya untuk mencapai akurasi, harap diketahui bahwa terjemahan otomatis mungkin mengandung kesalahan atau ketidakakuratan. Dokumen asli dalam bahasa aslinya harus dianggap sebagai sumber yang sah. Untuk informasi penting, disarankan menggunakan terjemahan profesional oleh manusia. Kami tidak bertanggung jawab atas kesalahpahaman atau penafsiran yang keliru yang timbul dari penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
